# télos MDLM: Master Overnight Training Suite
This notebook executes the target overnight training pipeline in sequence using native MLX Metal acceleration.

### Target Ratios & Pipeline:
1. **50M 1:20** (Upscaled from existing `phase_b_25m_1to20_mlx` checkpoint)
2. **25M 1:25** (Trained from scratch to 852M tokens)
3. **50M 1:25** (Upscaled from newly trained `phase_b_25m_1to25_mlx` checkpoint)

Memory GC is enforced on Metal GPU to keep RAM usage strictly under 8GB.


In [ ]:
import os
import sys
import time
import gc
import yaml
from pathlib import Path

# Ensure working directory is project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
from telos.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from telos.training.trainer import TelosMLXTrainer

def run_training_step(config_path, upscaled_source=None):
    print("=" * 85)
    print("STARTING OVERNIGHT RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if upscaled_source:
        src_ckpt, src_cfg = upscaled_source
        print("  [Net2Net] Upscaling model weights from: " + str(src_ckpt))
        load_upscaled_weights(model, cfg["model"], src_ckpt, src_cfg)
    
    trainer = TelosMLXTrainer(model, cfg)
    trainer.train()
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED RUN: " + str(config_path) + "\n")

# PIPELINE DEFINITION
start_time = time.time()

# 1. 50M 1:20 (Upscaled from 25M 1:20)
run_training_step(
    "configs/phase_b_50m_1to20_mlx.yaml",
    upscaled_source=("checkpoints/phase_b_25m_1to20_mlx/model.safetensors", "configs/phase_b_25m_1to20_mlx.yaml")
)

# 2. 25M 1:25 (From Scratch)
run_training_step("configs/phase_b_25m_1to25_mlx.yaml")

# 3. 50M 1:25 (Upscaled from new 25M 1:25)
run_training_step(
    "configs/phase_b_50m_1to25_mlx.yaml",
    upscaled_source=("checkpoints/phase_b_25m_1to25_mlx/checkpoint_final.safetensors", "configs/phase_b_25m_1to25_mlx.yaml")
)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"ALL OVERNIGHT RUNS COMPLETED SUCCESSFULLY IN {total_elapsed:.2f} HOURS!")
print("=" * 85)
